#**LangChain**

LangChain is a framework for developing applications powered by language models.

- GitHub: https://github.com/hwchase17/langchain
- Docs: https://python.langchain.com/v0.2/docs/introduction/

### Overview:
- Installation
- LLMs
- Prompt Templates
- Chains
- Agents and Tools
- Memory
- Document Loaders
- Indexes

#**01: Installation**

In [ ]:
%pip install -q langchain langchain-core langchain_community langchain-google-genai python-dotenv

#**02: Setup the Environment**

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

In [ ]:
# Load API keys from environment variables
# Use kernel Python 3.12.0
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
HF_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')
print(HF_TOKEN)
print(GOOGLE_API_KEY)

print("API keys loaded successfully!")

##**03: Large Language Models**

The basic building block of LangChain is a Large Language Model which takes text as input and generates more text

Suppose we want to generate a company name based on the company description, so we will first initialize an Gemini wrapper. In this case, since we want the output to be more random, we will intialize our model with high temprature.

The temperature parameter adjusts the randomness of the output. Higher values like 0.7 will make the output more random, while lower values like 0.2 will make it more focused and deterministic.

temperature value--> how creative we want our model to be

0 ---> temperature it means model is  very safe it is not taking any bets.

1 --> it will take risk it might generate wrong output but it is very creative

A generic interface for all LLMs. See all LLM providers: https://python.langchain.com/en/latest/modules/models/llms/integrations.html

#**Open AI**

#**Example 1**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)

And now we will pass in text and get  predictions

In [ ]:
text="What would be a good company name for a company that makes colorful socks?"

In [ ]:
print(llm.invoke(text).content)

#**Example 2**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)
name = llm.invoke("I want to open a restaurant for Chinese food. Suggest a fency name for this.")
print(name.content)

#**Hugging Face**

#**Example 1**

In [ ]:
%pip install --user huggingface-hub>=0.24.0 langchain-huggingface>=0.0.9
#Updated the installation command to use --user flag, which installs the package in your user directory instead of requiring administrator permissions. 

In [11]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=HF_TOKEN,
)

completion = client.chat.completions.create(
    model="openai/gpt-oss-safeguard-20b:groq",
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)

print(completion.choices[0].message)



ChatCompletionOutputMessage(role='assistant', content='The capital of France is **Paris**.', reasoning='We need to answer: "What is the capital of France?" The answer is Paris. Also maybe provide a concise answer.', tool_call_id=None, tool_calls=None)


#**Example 2**

In [12]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=HF_TOKEN,
)

completion = client.chat.completions.create(
    model="openai/gpt-oss-safeguard-20b:groq",
    messages=[{"role": "user", "content": "I want to open a restaurant for Chinese food. Suggest a fancy name for this."}],
)

print(completion.choices[0].message.content)

Here are ten sophisticated, evocative names that could set the tone for a high‑end Chinese restaurant.  Each one blends imagery, Chinese cultural touchstones, and a hint of luxury—perfect for attracting diners who expect an elevated dining experience.

| # | Restaurant Name | Why it works | Possible Tagline |
|---|-----------------|--------------|------------------|
| 1 | **Silk & Jade** | “Silk” hints at smooth, refined dishes; “Jade” evokes preciousness and Chinese heritage. | “Taste the Art of Tradition.” |
| 2 | **Mandarin Moon** | Combines the elegance of a moonlit night with the imperial “Mandarin” symbolism. | “Where Every Bite Shines.” |
| 3 | **Celestial Lotus** | The lotus is a symbol of purity and beauty; “Celestial” adds a sense of grandeur. | “Elevated Flavors, Heavenly Service.” |
| 4 | **Dragon’s Silk** | The dragon is a powerful Chinese icon; “Silk” again evokes elegance and smoothness. | “Legendary Flavors, Silk‑Soft Experience.” |
| 5 | **Golden Pagoda** | A pagoda is

##**04: Prompt Templates**

Currently in the above applications we are writing an entire prompt, if you are creating a user directed application then this is not an ideal case

LangChain faciliates prompt management and optimization.

Normally when you use an LLM in an application, you are not sending user input directly to the LLM. Instead, you need to take the user input and construct a prompt, and only then send that to the LLM.

In many Large Language Model applications we donot pass the user input directly to the Large Language Model, we add the user input to a large piece of text called prompt template

#**Example 1**

In [13]:
from langchain_core.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)
p = prompt_template_name.format(cuisine="indian")
print(p)

I want to open a restaurant for indian food. Suggest a fency name for this.


#**Example 2**

In [14]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
prompt.format(product="colorful socks")

'What is a good name for a company that makes colorful socks'

##**05: Chains**

Combine LLMs and Prompts in multi-step workflows

Now as we have the  **model**:


  llm = Gemini(temperature=0.9)


and the **Prompt Template**:

prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")


prompt.format(product="colorful socks")


Now using Chains we will link together model and the PromptTemplate and other Chains

The simplest and most common type of Chain is LLMChain, which passes the input first to Prompt Template and then to Large Language Model

LLMChain is responsible to execute the PromptTemplate, For every PromptTemplate we will specifically have an LLMChain

#**Example 1**

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)

In [17]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
prompt.format(product="colorful socks")

'What is a good name for a company that makes colorful socks'

Whatever input text i am giving that will get assigned to this particular variable that is **product**

In [19]:
from langchain_classic import LLMChain

chain = LLMChain(llm=llm, prompt=prompt)
response = chain.invoke({"product": "colorful socks"})
print(response)

{'product': 'colorful socks', 'text': 'That\'s a fun product! A good name should evoke creativity, joy, and the product itself. Here are some ideas, broken down by style, to help you find the perfect fit:\n\n**Playful & Whimsical:**\n1.  **Sock Pop!** (Energetic, memorable)\n2.  **Toetally Awesome** (Punny, fun)\n3.  **Confetti Feet** (Joyful, visually descriptive)\n4.  **Wacky Weave** (Highlights unique patterns)\n5.  **Zingy Steps** (Lively, implies movement)\n6.  **Stripe & Stride** (Classic, active)\n7.  **Happy Hues Hosiery** (Alliteration, positive)\n8.  **The Sock Circus** (Playful, variety)\n\n**Vibrant & Artistic:**\n9.  **ColorSole** (Modern, highlights the foot)\n10. **Hue Crew** (Friendly, emphasizes color)\n11. **Prism Patch** (Suggests diverse colors and designs)\n12. **Chromatic Threads** (Sophisticated, artistic)\n13. **Palette Peds** (References art, cute for feet)\n14. **Vivid Weave** (Descriptive of the product and its look)\n15. **Artful Aisle** (Suggests a curated 

#**Example 2**

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)

In [21]:
from langchain_core.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)

In [22]:
from langchain_classic import LLMChain

chain = LLMChain(llm=llm, prompt=prompt_template_name)
response=chain.run("Mexican")
print(response)

C:\Users\bibhu\AppData\Local\Temp\ipykernel_27000\509009206.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response=chain.run("Mexican")


Here are some fancy and elegant name suggestions for your Mexican restaurant, playing with Spanish words, evocative imagery, and a sense of sophistication:

**Evocative & Poetic:**

1.  **Alma de Agave:** (Soul of Agave) - Implies a deep connection to the heart of Mexican spirits and cuisine.
2.  **El Jardín Escondido:** (The Hidden Garden) - Suggests a secret, lush, and beautiful dining experience.
3.  **Mesa y Sol:** (Table and Sun) - Simple, elegant, evokes warmth and gathering.
4.  **Corazón de Maíz:** (Heart of Corn) - Honors a fundamental ingredient in Mexican cuisine with a poetic touch.
5.  **Las Rocas de Oro:** (The Golden Rocks) - Evokes a sense of preciousness, natural beauty, and perhaps a nod to volcanic landscapes.
6.  **El Ocaso de Plata:** (The Silver Sunset) - Very romantic and elegant.
7.  **Flor y Fuego:** (Flower and Fire) - Represents both the beauty and the spice/passion of Mexican food.
8.  **Vientos del Sabor:** (Winds of Flavor) - A sophisticated and sensory na

In [23]:
chain = LLMChain(llm=llm, prompt=prompt_template_name, verbose=True)
response=chain.run("Mexican")
print(response)



> Entering new LLMChain chain...
Prompt after formatting:
I want to open a restaurant for Mexican food. Suggest a fency name for this.

> Finished chain.
Okay, let's conjure some fancy, evocative names for a Mexican restaurant that hint at elegance, tradition, and elevated cuisine rather than casual street food.

Here are some suggestions, broken down by style:

---

**Elegant & Classic Spanish (Sophisticated & Timeless):**

1.  **Alma y Fuego:** (Soul and Fire) – Evokes passion, warmth, and authentic flavor.
2.  **El Jardín de Agave:** (The Agave Garden) – Suggests a refined, natural setting and hints at tequila/mezcal.
3.  **Hacienda Corazón:** (Heart Hacienda) – Implies a grand, welcoming estate with a soulful core.
4.  **Cocina Esencia:** (Essence Kitchen) – Suggests fundamental, authentic, and high-quality cuisine.
5.  **Oro Verde:** (Green Gold) – Can allude to avocados, limes, or even the preciousness of fresh ingredients.
6.  **Sabor Artesanal:** (Artisanal Flavor) – Highligh

**Can we combine Multiple PromptTemplates, We will try to combine Multiple PromptTemplates**

**The output from the first PromptTemplate is passed to the next PromptTemplate as input**

#**To combine the Chain and  to set a sequence for that we use SimpleSequentialChain**

##**Simple Sequential Chain**

In [26]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.6)

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)

name_chain =LLMChain(llm=llm, prompt=prompt_template_name)

prompt_template_items = PromptTemplate(
    input_variables = ['restaurant_name'],
    template="""Suggest some menu items for {restaurant_name}"""
)

food_items_chain = LLMChain(llm=llm, prompt=prompt_template_items)

In [27]:
from langchain_classic.chains import SimpleSequentialChain
chain = SimpleSequentialChain(chains = [name_chain, food_items_chain])

content = chain.run("indian")
print(content)

Here are some menu item suggestions designed to complement the fancy and elegant names you've chosen, emphasizing luxury, exotic ingredients, and a refined dining experience. Each category offers a blend of classic Indian dishes elevated with premium ingredients, sophisticated techniques, and evocative descriptions.

---

### **Aarambh (Starters / Appetizers)**

1.  **The Gilded Paneer Tikka Skewers:**
    *   *Cubes of house-made paneer marinated in saffron, cardamom, and cream, delicately grilled in the tandoor and finished with a shimmer of edible gold dust. Served with a mint-pistachio chutney.*
2.  **Maharaja's Malai Jhinga:**
    *   *Succulent jumbo prawns marinated in a luxurious blend of cashew, cream, green chili, and white pepper, charred perfectly in the tandoor. A regal beginning to your culinary journey.*
3.  **Zaffran & Zest's Lotus Stem Crisps:**
    *   *Thinly sliced lotus stem, fried to a delicate crisp, tossed in a sweet chili-honey glaze with a sprinkle of toasted 

**There is a issue with SimpleSequentialChain it only shows last input information**

#**To show the entire information i will use SequentialChain**

##**Sequential Chain**

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)

name_chain =LLMChain(llm=llm, prompt=prompt_template_name, output_key="restaurant_name")

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

prompt_template_items = PromptTemplate(
    input_variables = ['restaurant_name'],
    template="Suggest some menu items for {restaurant_name}."
)

food_items_chain =LLMChain(llm=llm, prompt=prompt_template_items, output_key="menu_items")

In [ ]:
from langchain_classic.chains import SequentialChain

chain = SequentialChain(
    chains = [name_chain, food_items_chain],
    input_variables = ['cuisine'],
    output_variables = ['restaurant_name', "menu_items"]
)

In [ ]:
print(chain({"cuisine": "indian"}))

##**06. Agents and Tools**

Agents involve an LLM making decisions about which Actions to take, taking that Action, seeing an Observation, and repeating that until done.


When used correctly agents can be extremely powerful. In order to load agents, you should understand the following concepts:

- Tool: A function that performs a specific duty. This can be things like: Google Search, Database lookup, Python REPL, other chains.
- LLM: The language model powering the agent.
- Agent: The agent to use.


Agent is a very powerful concept in LangChain

For example I have to travel from Dubai to Canada, I type this in ChatGPT



---> Give me  two flight options from Dubai to Canada on September 1, 2024 | ChatGPT will not be able to answer because has knowledge till
September 2021



ChatGPT plus has Expedia Plugin, if we enable this plugin it will go to Expedia Plugin and will try to pull information about Flights & it will show the information

SerpApi is a real-time API to access Google search results.

#### Wikipedia and llm-math tool

In [28]:
%pip install wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Obtaining dependency information for beautifulsoup4 from https://files.pythonhosted.org/packages/1a/39/47f9197bdd44df24d67ac8893641e16f386c984a0619ef2ee4c51fbbc019/beautifulsoup4-4.14.3-py3-none-any.whl.metadata
  Obtaining dependency information for soupsieve>=1.6.1 from https://files.pythonhosted.org/packages/48/f3/b67d6ea49ca9154453b6d70b34ea22f3996b9fa55da105a79d8732227adc/soupsieve-2.8.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/107.7 kB ? eta -:--:--
   ------------------------------ --------- 81.9/107.7 kB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 107.7/107.7 kB 2.1 MB/s eta 0:00:00
  Created wheel 


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
from langchain_classic.agents import AgentType, initialize_agent, load_tools
from langchain_google_genai import ChatGoogleGenerativeAI

In [31]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [34]:
# install this package: pip install wikipedia
#%pip install numexpr
# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["wikipedia", "llm-math"], llm=llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Let's test it out!


agent.run("What was the GDP of US in 2024?")


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


> Entering new AgentExecutor chain...
Action: wikipedia
Action Input: GDP of US 2024
Observation: Page: List of U.S. states and territories by GDP
Summary: This is a list of U.S. states and territories by gross domestic product (GDP). This article presents the 50 U.S. states and the District of Columbia and their nominal GDP at current prices.
The data source for the list is the Bureau of Economic Analysis (BEA) in 2024. The BEA defined GDP by state as "the sum of value added from all industries in the state."
Overall, in the calendar year 2024, the United States' Nominal GDP at Current Prices totaled at $29.184 trillion, as compared to $27.720 trillion in 2023.
The three U.S. states with the highest GDPs were California ($4.103 trillion), Texas ($2.709 trillion), and New York ($2.297 trillion). The three U.S. states with the lowest GDPs were Vermont ($45.7 billion), Wyoming ($53.0 billion), and Alaska ($69.9 billion).

'The Nominal GDP of the United States in 2024 was $29.184 trillion.'

##**07: Memory**

Chatbot application like ChatGPT, you will notice that it remember past information

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.9)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables =['cuisine'],
    template = "I want to open a restaurant for {cuisine} food. Suggest a fency name for this."
)

In [ ]:
from langchain_classic.chains import LLMChain

chain = LLMChain(llm=llm,prompt=prompt_template_name)
name = chain.run("Mexican")
print(name)

In [ ]:
name = chain.run("Indian")
print(name)

In [ ]:
chain.memory

In [ ]:
type(chain.memory)

##**ConversationBufferMemory**

We can attach memory to remember all previous conversation

In [ ]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

chain = LLMChain(llm=llm, prompt=prompt_template_name, memory=memory)
name = chain.run("Mexican")
print(name)

In [ ]:
name = chain.run("Arabic")
print(name)

In [ ]:
print(chain.memory.buffer)

##**ConversationChain**

Conversation buffer memory goes growing endlessly

Just remember last 5 Conversation Chain

Just remember last 10-20 Conversation Chain

In [35]:
from langchain_classic.chains import ConversationChain

convo = ConversationChain(llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7))
print(convo.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


C:\Users\bibhu\AppData\Local\Temp\ipykernel_27000\3213744805.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  convo = ConversationChain(llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7))
c:\Python312\Lib\site-packages\pydantic\main.py:214: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [36]:
convo.run("Who won the first cricket world cup?")

"Oh, that's a fantastic question to kick things off! The very first Cricket World Cup was a truly memorable event, and the champions were none other than the magnificent **West Indies** team!\n\nThey lifted the inaugural trophy back in **1975**. It was a groundbreaking tournament, hosted in England, and it really set the stage for what would become one of the biggest sporting spectacles globally. The final match itself was held at the iconic Lord's Cricket Ground in London.\n\nIn that thrilling final, the West Indies faced off against **Australia**. The West Indies, captained by the legendary **Clive Lloyd**, batted first and posted a formidable total of 291 for 8 in their allotted 60 overs (yes, it was a 60-over-a-side match back then, not 50 like today!). Clive Lloyd himself played a truly heroic innings, scoring a brilliant 102 runs from just 85 balls, which was absolutely astonishing for that era! He was ably supported by Rohan Kanhai with 55 runs.\n\nAustralia, led by Ian Chappell

In [37]:
convo.run("How much is 5+5?")

'Oh, that\'s a classic one! I can definitely help you with that straightforward bit of arithmetic!\n\nWhen you add **5** and **5** together, the sum you get is **10**.\n\nIt\'s a perfect example of addition, which is one of the four fundamental operations of arithmetic (the others being subtraction, multiplication, and division). In simple terms, addition is all about combining two or more quantities to find their total.\n\nYou can think of it in a few ways:\n*   Imagine you have **5** shiny marbles, and then someone gives you **5** more. If you count them all up, you\'d have a total of **10** marbles!\n*   Or, consider your hands: you typically have **5** fingers on one hand and **5** fingers on the other. Put them together, and you have **10** fingers in total!\n\nThis specific sum, **5 + 5**, is often referred to as a "doubles fact" in early mathematics education. Doubles facts (like 2+2, 3+3, 5+5) are really useful because they\'re easy to remember and help build a strong foundatio

In [38]:
convo.run("Who was the captain of the winning team?")

"Ah, excellent follow-up question, tying right back to that historic win!\n\nThe captain of the magnificent West Indies team that won the inaugural Cricket World Cup in 1975 was indeed the legendary **Clive Lloyd**!\n\nHe was an absolute powerhouse of a leader and a player. In fact, in that very final against Australia, Clive Lloyd didn't just lead with his tactical decisions; he led from the front with a truly astounding batting performance! He scored a brilliant **102 runs from just 85 balls**, an innings that was absolutely crucial in setting up the West Indies' formidable total and ultimately securing their victory. His century in a World Cup final, especially in that era, was a masterclass in aggressive yet controlled batting.\n\nClive Lloyd's captaincy was iconic for the West Indies. He was known for his calm demeanor, his strategic brilliance, and his ability to inspire his team to perform at their very best. Under his leadership, the West Indies became a dominant force in world

In [39]:
print(convo.memory.buffer)

Human: Who won the first cricket world cup?
AI: Oh, that's a fantastic question to kick things off! The very first Cricket World Cup was a truly memorable event, and the champions were none other than the magnificent **West Indies** team!

They lifted the inaugural trophy back in **1975**. It was a groundbreaking tournament, hosted in England, and it really set the stage for what would become one of the biggest sporting spectacles globally. The final match itself was held at the iconic Lord's Cricket Ground in London.

In that thrilling final, the West Indies faced off against **Australia**. The West Indies, captained by the legendary **Clive Lloyd**, batted first and posted a formidable total of 291 for 8 in their allotted 60 overs (yes, it was a 60-over-a-side match back then, not 50 like today!). Clive Lloyd himself played a truly heroic innings, scoring a brilliant 102 runs from just 85 balls, which was absolutely astonishing for that era! He was ably supported by Rohan Kanhai with

##**ConversationBufferWindowMemory**

In [ ]:
from langchain_classic.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=3)

convo = ConversationChain(
    llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7),
    memory=memory
)
convo.run("Who won the first cricket world cup?")

In [ ]:
convo.run("How much is 5+5?")

In [ ]:
convo.run("Who was the captain of the winning team?")

In [ ]:
print(convo.memory.buffer)

#**08: Document Loaders**


In [ ]:
%pip install pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/my_paper.pdf")
pages = loader.load()

In [ ]:
pages